> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop.

# VOD Analysis Pipeline

S3에 비디오가 업로드되면 EventBridge를 통해 3개의 Lambda 함수가 병렬로 실행되는 파이프라인을 구축합니다.

```
S3 (비디오 업로드)  →  EventBridge Rule  →  Lambda: Marengo 임베딩 생성
                                         →  Lambda: Pegasus 요약 생성
                                         →  Lambda: 메타데이터 저장 (DynamoDB)
```

## Part 0: Setup

### Dependencies

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import boto3, botocore, json, time, io, zipfile, re
from boto3.dynamodb.conditions import Key
from IPython.display import HTML, display

### Configure AWS

In [ ]:
session = boto3.Session()
AWS_REGION = session.region_name

workshop_supported_regions = ["us-east-1", "eu-west-1", "ap-northeast-2"]
if AWS_REGION not in workshop_supported_regions:
    raise ValueError(f"Region {AWS_REGION} is not supported. Use one of: {workshop_supported_regions}")

bedrock_client = session.client('bedrock-runtime')
s3_client = session.client('s3')
dynamodb_client = session.client('dynamodb')
dynamodb_resource = session.resource('dynamodb')
lambda_client = session.client('lambda')
iam_client = session.client('iam')
events_client = session.client('events')
sts_client = session.client('sts')

AWS_ACCOUNT_ID = sts_client.get_caller_identity()["Account"]
print(f"Region: {AWS_REGION}, Account: {AWS_ACCOUNT_ID}")

### Configure S3 bucket

In [ ]:
S3_BUCKET_NAME = "<YOUR_S3_BUCKET_NAME>"  # TODO: Replace with your S3 bucket name

if S3_BUCKET_NAME == "<YOUR_S3_BUCKET_NAME>" or S3_BUCKET_NAME == "":
    raise ValueError("Please replace <YOUR_S3_BUCKET_NAME> with your S3 bucket name")

# Verify bucket access
try:
    s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
    print(f"S3 Bucket: {S3_BUCKET_NAME}")
except Exception as e:
    raise ValueError(f"Cannot access bucket: {e}")

### Configure Model IDs

In [ ]:
# Marengo: base model ID for StartAsyncInvoke
MARENGO_MODEL_ID = 'twelvelabs.marengo-embed-3-0-v1:0'

# Pegasus: inference profile ID for InvokeModel (region-specific)
PEGASUS_INFERENCE_ID_REGIONS = {
    "us-east-1": "us.twelvelabs.pegasus-1-2-v1:0",
    "eu-west-1": "eu.twelvelabs.pegasus-1-2-v1:0",
    "ap-northeast-2": "apac.twelvelabs.pegasus-1-2-v1:0"
}
PEGASUS_MODEL_ID = PEGASUS_INFERENCE_ID_REGIONS[AWS_REGION]

print(f"Marengo: {MARENGO_MODEL_ID}")
print(f"Pegasus: {PEGASUS_MODEL_ID}")

### Resource Names

In [ ]:
PIPELINE_PREFIX = "pipeline"
TABLE_NAME = "vod-pipeline-tasks"
ROLE_NAME = "vod-pipeline-lambda-role"
RULE_NAME = "vod-pipeline-video-upload"
LAMBDA_NAMES = {
    "embedding": "vod-pipeline-embedding",
    "summary": "vod-pipeline-summary",
    "metadata": "vod-pipeline-metadata"
}

print(f"DynamoDB Table: {TABLE_NAME}")
print(f"IAM Role: {ROLE_NAME}")
print(f"EventBridge Rule: {RULE_NAME}")
print(f"Lambda Functions: {list(LAMBDA_NAMES.values())}")

---
## Part 1: Create Infrastructure

### 1a. DynamoDB Table

In [ ]:
try:
    dynamodb_client.create_table(
        TableName=TABLE_NAME,
        KeySchema=[
            {"AttributeName": "video_key", "KeyType": "HASH"},
            {"AttributeName": "task_type", "KeyType": "RANGE"}
        ],
        AttributeDefinitions=[
            {"AttributeName": "video_key", "AttributeType": "S"},
            {"AttributeName": "task_type", "AttributeType": "S"}
        ],
        BillingMode="PAY_PER_REQUEST"
    )
    dynamodb_client.get_waiter('table_exists').wait(TableName=TABLE_NAME)
    print(f"Table '{TABLE_NAME}' created")
except dynamodb_client.exceptions.ResourceInUseException:
    print(f"Table '{TABLE_NAME}' already exists")

### 1b. IAM Role for Lambda

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    role_response = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="VOD pipeline Lambda execution role"
    )
    ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Role created: {ROLE_ARN}")
except iam_client.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = iam_client.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print(f"Role already exists: {ROLE_ARN}")

# Attach inline policy
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:*:*:*"
        },
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:StartAsyncInvoke", "bedrock:GetAsyncInvoke"],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": [f"arn:aws:s3:::{S3_BUCKET_NAME}", f"arn:aws:s3:::{S3_BUCKET_NAME}/*"]
        },
        {
            "Effect": "Allow",
            "Action": ["dynamodb:PutItem", "dynamodb:UpdateItem", "dynamodb:GetItem"],
            "Resource": f"arn:aws:dynamodb:{AWS_REGION}:{AWS_ACCOUNT_ID}:table/{TABLE_NAME}"
        }
    ]
}

iam_client.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="vod-pipeline-policy",
    PolicyDocument=json.dumps(inline_policy)
)
print("Inline policy attached")
print("Waiting for IAM role propagation...")
time.sleep(10)

### 1c. Lambda Functions

3개의 Lambda 함수를 생성합니다:
- **embedding**: Marengo `StartAsyncInvoke`로 임베딩 생성 시작
- **summary**: Pegasus `InvokeModel`로 비디오 요약 생성
- **metadata**: 비디오 메타데이터를 DynamoDB에 저장

In [ ]:
# Helper: create zip from code string
def create_lambda_zip(code_str):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('lambda_function.py', code_str)
    buf.seek(0)
    return buf.read()

# Lambda code definitions
EMBEDDING_CODE = '''import boto3, json, os, uuid, time
from decimal import Decimal
from botocore.config import Config

bedrock = boto3.client("bedrock-runtime", config=Config(signature_version="v4"))
dynamodb = boto3.resource("dynamodb")
table = dynamodb.Table(os.environ["TABLE_NAME"])
MODEL_ID = os.environ["MARENGO_MODEL_ID"]

def handler(event, context):
    bucket = event["detail"]["bucket"]["name"]
    key = event["detail"]["object"]["key"]
    account_id = context.invoked_function_arn.split(":")[4]
    task_id = str(uuid.uuid4())[:8]

    response = bedrock.start_async_invoke(
        modelId=MODEL_ID,
        modelInput={
            "inputType": "video",
            "video": {
                "mediaSource": {
                    "s3Location": {"uri": f"s3://{bucket}/{key}", "bucketOwner": account_id}
                },
                "embeddingOption": ["visual"],
                "embeddingScope": ["clip"]
            }
        },
        outputDataConfig={
            "s3OutputDataConfig": {"s3Uri": f"s3://{bucket}/pipeline/embeddings/{task_id}/"}
        }
    )

    table.put_item(Item={
        "video_key": key, "task_type": "embedding",
        "task_id": task_id, "invocation_arn": response["invocationArn"],
        "status": "processing", "timestamp": Decimal(str(int(time.time())))
    })
    return {"statusCode": 200, "taskId": task_id}
'''

SUMMARY_CODE = '''import boto3, json, os, time
from decimal import Decimal
from botocore.config import Config

bedrock = boto3.client("bedrock-runtime", config=Config(signature_version="v4"))
dynamodb = boto3.resource("dynamodb")
table = dynamodb.Table(os.environ["TABLE_NAME"])
MODEL_ID = os.environ["PEGASUS_MODEL_ID"]

def handler(event, context):
    bucket = event["detail"]["bucket"]["name"]
    key = event["detail"]["object"]["key"]
    account_id = context.invoked_function_arn.split(":")[4]

    try:
        response = bedrock.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps({
                "inputPrompt": "Summarize this video in 3 sentences.",
                "mediaSource": {
                    "s3Location": {
                        "uri": f"s3://{bucket}/{key}",
                        "bucketOwner": account_id
                    }
                },
                "temperature": 0
            }),
            contentType="application/json",
            accept="application/json"
        )
        result = json.loads(response["body"].read())
        summary = result.get("message", "")
        status = "completed"
    except Exception as e:
        summary = str(e)
        status = "failed"

    table.put_item(Item={
        "video_key": key, "task_type": "summary",
        "summary": summary, "status": status,
        "timestamp": Decimal(str(int(time.time())))
    })
    return {"statusCode": 200}
'''

METADATA_CODE = '''import boto3, os, time
from decimal import Decimal

dynamodb = boto3.resource("dynamodb")
table = dynamodb.Table(os.environ["TABLE_NAME"])

def handler(event, context):
    bucket = event["detail"]["bucket"]["name"]
    key = event["detail"]["object"]["key"]
    size = event["detail"]["object"].get("size", 0)

    table.put_item(Item={
        "video_key": key, "task_type": "metadata",
        "bucket": bucket, "size_bytes": Decimal(str(size)),
        "size_mb": Decimal(str(round(size / (1024 * 1024), 2))),
        "upload_time": Decimal(str(int(time.time()))),
        "status": "completed"
    })
    return {"statusCode": 200}
'''

print("Lambda code defined")

In [ ]:
# Create or update Lambda functions
lambda_configs = {
    "embedding": {
        "code": EMBEDDING_CODE,
        "timeout": 60,
        "memory": 128,
        "env": {"TABLE_NAME": TABLE_NAME, "MARENGO_MODEL_ID": MARENGO_MODEL_ID}
    },
    "summary": {
        "code": SUMMARY_CODE,
        "timeout": 900,
        "memory": 256,
        "env": {"TABLE_NAME": TABLE_NAME, "PEGASUS_MODEL_ID": PEGASUS_MODEL_ID}
    },
    "metadata": {
        "code": METADATA_CODE,
        "timeout": 30,
        "memory": 128,
        "env": {"TABLE_NAME": TABLE_NAME}
    }
}

lambda_arns = {}

for key, config in lambda_configs.items():
    func_name = LAMBDA_NAMES[key]
    zip_bytes = create_lambda_zip(config["code"])

    try:
        resp = lambda_client.create_function(
            FunctionName=func_name,
            Runtime="python3.12",
            Role=ROLE_ARN,
            Handler="lambda_function.handler",
            Code={"ZipFile": zip_bytes},
            Timeout=config["timeout"],
            MemorySize=config["memory"],
            Environment={"Variables": config["env"]}
        )
        lambda_arns[key] = resp["FunctionArn"]
        print(f"Created: {func_name}")
    except lambda_client.exceptions.ResourceConflictException:
        # Update code first, then wait for it to finish before updating config
        lambda_client.update_function_code(FunctionName=func_name, ZipFile=zip_bytes)
        waiter = lambda_client.get_waiter('function_updated_v2')
        waiter.wait(FunctionName=func_name)
        lambda_client.update_function_configuration(
            FunctionName=func_name,
            Timeout=config["timeout"],
            MemorySize=config["memory"],
            Environment={"Variables": config["env"]},
            Role=ROLE_ARN
        )
        resp = lambda_client.get_function(FunctionName=func_name)
        lambda_arns[key] = resp["Configuration"]["FunctionArn"]
        print(f"Updated: {func_name}")

print(f"\nAll Lambda functions ready: {list(lambda_arns.keys())}")

### 1d. EventBridge Configuration

S3 버킷에서 EventBridge 알림을 활성화하고, `pipeline/videos/` 경로에 업로드된 파일에 대해 3개의 Lambda를 병렬 트리거하는 Rule을 생성합니다.

In [ ]:
# Enable EventBridge notifications on S3 bucket (preserve existing config)
current_config = s3_client.get_bucket_notification_configuration(Bucket=S3_BUCKET_NAME)
current_config.pop('ResponseMetadata', None)
current_config['EventBridgeConfiguration'] = {}
s3_client.put_bucket_notification_configuration(
    Bucket=S3_BUCKET_NAME,
    NotificationConfiguration=current_config
)
print(f"S3 EventBridge notification enabled for '{S3_BUCKET_NAME}'")

In [ ]:
# Create EventBridge rule
event_pattern = {
    "source": ["aws.s3"],
    "detail-type": ["Object Created"],
    "detail": {
        "bucket": {"name": [S3_BUCKET_NAME]},
        "object": {"key": [{"prefix": f"{PIPELINE_PREFIX}/videos/"}]}
    }
}

rule_response = events_client.put_rule(
    Name=RULE_NAME,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="Trigger VOD pipeline on video upload"
)
RULE_ARN = rule_response["RuleArn"]
print(f"EventBridge rule created: {RULE_NAME}")
print(f"Pattern: {PIPELINE_PREFIX}/videos/* -> 3 Lambda functions")

In [ ]:
# Add Lambda permissions for EventBridge and set targets
targets = []
for key, func_arn in lambda_arns.items():
    func_name = LAMBDA_NAMES[key]
    # Allow EventBridge to invoke this Lambda
    try:
        lambda_client.add_permission(
            FunctionName=func_name,
            StatementId=f"eventbridge-{RULE_NAME}",
            Action="lambda:InvokeFunction",
            Principal="events.amazonaws.com",
            SourceArn=RULE_ARN
        )
    except lambda_client.exceptions.ResourceConflictException:
        pass  # Permission already exists

    targets.append({"Id": f"target-{key}", "Arn": func_arn})

events_client.put_targets(Rule=RULE_NAME, Targets=targets)
print(f"Targets configured: {[t['Id'] for t in targets]}")
print("\nPipeline infrastructure ready!")

---
## Part 2: Test Pipeline

### 2a. Upload Video

`pipeline/videos/` 경로에 비디오를 업로드하여 파이프라인을 트리거합니다. 기존에 다운로드한 Netflix Open Content 비디오가 있으면 재사용하고, 없으면 새로 다운로드합니다.

In [ ]:
# Check if video already exists in bucket from previous workshops
VIDEO_FILENAME = "Sparks_4096x2160_5994fps_SDR.mp4"
PIPELINE_VIDEO_KEY = f"{PIPELINE_PREFIX}/videos/{VIDEO_FILENAME}"
SOURCE_VIDEO_KEY = f"videos/{VIDEO_FILENAME}"
NETFLIX_S3_URI = "s3://download.opencontent.netflix.com/TechblogAssets/Sparks/encodes/Sparks_4096x2160_5994fps_SDR.mp4"

def parse_s3_uri(s3_uri):
    match = re.match(r'^s3://([^/]+)/(.+)$', s3_uri)
    if not match:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")
    return match.group(1), match.group(2)

# Try to copy from existing location in bucket
video_ready = False
try:
    s3_client.head_object(Bucket=S3_BUCKET_NAME, Key=SOURCE_VIDEO_KEY)
    print(f"Found existing video: {SOURCE_VIDEO_KEY}")
    print(f"Copying to {PIPELINE_VIDEO_KEY}...")
    s3_client.copy_object(
        Bucket=S3_BUCKET_NAME,
        CopySource={"Bucket": S3_BUCKET_NAME, "Key": SOURCE_VIDEO_KEY},
        Key=PIPELINE_VIDEO_KEY
    )
    video_ready = True
    print(f"Copied to {PIPELINE_VIDEO_KEY}")
except:
    print("No existing video found. Downloading from Netflix Open Content...")
    src_bucket, src_key = parse_s3_uri(NETFLIX_S3_URI)
    anon_s3 = boto3.client('s3', config=botocore.client.Config(signature_version=botocore.UNSIGNED))
    data = anon_s3.get_object(Bucket=src_bucket, Key=src_key)['Body'].read()
    s3_client.put_object(Bucket=S3_BUCKET_NAME, Key=PIPELINE_VIDEO_KEY, Body=data)
    video_ready = True
    print(f"Uploaded to {PIPELINE_VIDEO_KEY}")

if video_ready:
    print(f"\nPipeline triggered! EventBridge will invoke 3 Lambda functions in parallel.")

### 2b. View the Video

In [ ]:
def play_video(video_url, start_time=0):
    display(HTML(f'<video width="640" controls><source src="{video_url}#t={start_time}" type="video/mp4"></video>'))

presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": S3_BUCKET_NAME, "Key": PIPELINE_VIDEO_KEY},
    ExpiresIn=3600
)
play_video(presigned_url)

### 2c. Monitor Results

DynamoDB 테이블을 폴링하여 3개의 Lambda 실행 결과를 확인합니다. 메타데이터와 임베딩은 빠르게 완료되고, Pegasus 요약은 비디오 길이에 따라 수 분이 소요될 수 있습니다.

In [ ]:
table = dynamodb_resource.Table(TABLE_NAME)

print(f"Waiting for pipeline results (video: {PIPELINE_VIDEO_KEY})...\n")

expected_tasks = {"metadata", "embedding", "summary"}
completed_tasks = set()
max_wait = 600  # 10 minutes max
poll_start = time.time()

while completed_tasks != expected_tasks and (time.time() - poll_start) < max_wait:
    response = table.query(
        KeyConditionExpression=Key('video_key').eq(PIPELINE_VIDEO_KEY)
    )

    for item in response.get('Items', []):
        task_type = item['task_type']
        if task_type not in completed_tasks:
            elapsed = int(time.time() - poll_start)
            print(f"  [{elapsed:3d}s] {task_type}: {item.get('status', 'unknown')}")
            completed_tasks.add(task_type)

    if completed_tasks != expected_tasks:
        remaining = expected_tasks - completed_tasks
        print(f"        ... waiting for: {remaining}")
        time.sleep(10)

elapsed = int(time.time() - poll_start)
if completed_tasks == expected_tasks:
    print(f"\nAll tasks completed in {elapsed}s!")
else:
    missing = expected_tasks - completed_tasks
    print(f"\nTimeout after {elapsed}s. Missing: {missing}")
    print("The summary task may still be running. Check results below.")

### 2d. View Results

In [ ]:
# Query all results for the video
response = table.query(
    KeyConditionExpression=Key('video_key').eq(PIPELINE_VIDEO_KEY)
)

for item in response.get('Items', []):
    task_type = item['task_type']
    status = item.get('status', 'unknown')
    print(f"{'='*60}")
    print(f"Task: {task_type} | Status: {status}")
    print(f"{'='*60}")

    if task_type == 'metadata':
        print(f"  Bucket:      {item.get('bucket', 'N/A')}")
        print(f"  Size:        {item.get('size_mb', 'N/A')} MB")
        print(f"  Upload time: {item.get('upload_time', 'N/A')}")

    elif task_type == 'embedding':
        print(f"  Task ID:        {item.get('task_id', 'N/A')}")
        print(f"  Invocation ARN: {item.get('invocation_arn', 'N/A')}")

    elif task_type == 'summary':
        print(f"  Summary:")
        print(f"  {item.get('summary', 'N/A')}")

    print()

### 2e. Check Embedding Job Status

In [ ]:
# Get embedding task info from DynamoDB
response = table.get_item(
    Key={"video_key": PIPELINE_VIDEO_KEY, "task_type": "embedding"}
)

item = response.get('Item')
if item and 'invocation_arn' in item:
    invocation_arn = item['invocation_arn']
    job = bedrock_client.get_async_invoke(invocationArn=invocation_arn)
    status = job['status']
    print(f"Embedding job status: {status}")

    if status == "Completed":
        # Retrieve embedding results from S3
        task_id = item['task_id']
        prefix = f"{PIPELINE_PREFIX}/embeddings/{task_id}/"
        objs = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=prefix)

        for obj in objs.get('Contents', []):
            if obj['Key'].endswith('output.json'):
                content = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=obj['Key'])['Body'].read().decode()
                data = json.loads(content).get('data', [])
                print(f"Embedding segments: {len(data)}")
                if data:
                    print(f"Embedding dimension: {len(data[0]['embedding'])}")
                    print(f"\nFirst 3 segments:")
                    for seg in data[:3]:
                        print(f"  {seg['startSec']:.1f}s - {seg['endSec']:.1f}s | {seg.get('embeddingOption', 'N/A')}")
                break

        # Update DynamoDB status
        table.update_item(
            Key={"video_key": PIPELINE_VIDEO_KEY, "task_type": "embedding"},
            UpdateExpression="SET #s = :s",
            ExpressionAttributeNames={"#s": "status"},
            ExpressionAttributeValues={":s": "completed"}
        )
    elif status == "InProgress":
        print("Embedding is still processing. Re-run this cell later.")
    else:
        print(f"Job details: {json.dumps(job, indent=2, default=str)}")
else:
    print("No embedding task found. The Lambda may not have triggered yet.")

---
## Part 3: Cleanup

In [ ]:
# Remove EventBridge targets and rule
try:
    events_client.remove_targets(Rule=RULE_NAME, Ids=[f"target-{k}" for k in LAMBDA_NAMES])
    events_client.delete_rule(Name=RULE_NAME)
    print(f"EventBridge rule '{RULE_NAME}' deleted")
except Exception as e:
    print(f"EventBridge cleanup: {e}")

In [ ]:
# Delete Lambda functions
for key, func_name in LAMBDA_NAMES.items():
    try:
        lambda_client.delete_function(FunctionName=func_name)
        print(f"Lambda '{func_name}' deleted")
    except Exception as e:
        print(f"Lambda '{func_name}': {e}")

In [ ]:
# Delete IAM role (detach policy first)
try:
    iam_client.delete_role_policy(RoleName=ROLE_NAME, PolicyName="vod-pipeline-policy")
    iam_client.delete_role(RoleName=ROLE_NAME)
    print(f"IAM role '{ROLE_NAME}' deleted")
except Exception as e:
    print(f"IAM cleanup: {e}")

In [ ]:
# Delete DynamoDB table
try:
    dynamodb_client.delete_table(TableName=TABLE_NAME)
    print(f"DynamoDB table '{TABLE_NAME}' deleted")
except Exception as e:
    print(f"DynamoDB cleanup: {e}")

In [ ]:
# Clean up pipeline files in S3
try:
    response = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=f"{PIPELINE_PREFIX}/")
    if 'Contents' in response:
        objects = [{'Key': obj['Key']} for obj in response['Contents']]
        s3_client.delete_objects(Bucket=S3_BUCKET_NAME, Delete={'Objects': objects})
        print(f"Deleted {len(objects)} pipeline files from S3")
    else:
        print("No pipeline files to clean up")
except Exception as e:
    print(f"S3 cleanup: {e}")